# Tables of initial carbon values for Summer's box model


### Import

In [1]:
import numpy as np
import numpy.matlib
import numpy.ma as ma

import xarray as xr
#xr.set_options(enable_cftimeindex=True)
from xarray.coding.times import CFTimedeltaCoder

import time
import cftime
import netCDF4 as nc
from datetime import timedelta

import pandas as pd

#import xcdat

import matplotlib as mpl
import matplotlib.pyplot as plt

#import cartopy.crs as ccrs
#from cartopy.util import add_cyclic_point

import glob


## notes on packages to add to this kernel
import nc_time_axis
import cartopy.crs as ccrs
import cartopy

# custom functions for loading full data fields
from loading_function_flat10 import load_one_model_onevar, load_grid, select_time_slice, weighted_temporal_mean, load_observations 


### Set up lists

In [55]:
outputdir= '/glade/campaign/cgd/tss/people/aswann/flat10/'

# modellist_orig= ['ACCESS-ESM1-5',  
#             'CESM2',    
#             'GFDL-ESM4',  
#             'GISS_E2.1',  
#             'NorESM2-LM',
#             'MPI-ESM1-2-LR',
#             'CNRM-ESM2-1',
#             'HadCM3LC-Bris',
#             'UKESM1.2']
# modellist=modellist_orig

runlist = ['flat10','flat10_zec','flat10_cdr']
# use a wildcard to capture different ways the folders and runs are named across models
runlist_wc = ['*lat10','*zec','*cdr']

varlist_load=['cVeg','cSoil','cLitter','nbp'] #, 'gpp','fgco2', 'ra', 'rh']#, 'npp'] # not working beyond nbp for norESM
#varlist_analyze=['cVeg','cSoil','cTot','cLitter','nbp','gpp','rh','tas','pr']
varlist_analyze=['cVeg','cSoil','cTot','cLitter','nbp']
varlist=varlist_load
unitslist=['kgC m-2','kgC m-2','kgC m-2','kgC m-2 s-1','kgC m-2 s-1','kgC m-2 s-1']

# there seems to be a problem with ra for NorESM

modelcolors=['tab:blue','tab:orange','tab:green','tab:red','tab:gray','tab:purple','tab:cyan','gold','tab:brown']

# to match Sanderson paper:
modellist_ben=['ACCESS-ESM1-5',
                'CESM2',
                'GFDL-ESM4',
                'NASA-GISS-E2.1-G-CC2', #'GISS_E2.1',
                'NorESM2-LM',
                'MPI-ESM1-2-LR',
                'CNRM-ESM2-1',
                'UKESM1.2',
                # 'UVic-ESCM-2-10',
                'HadCM3LC-Bris',
                'MIROC-ES2L']
# bencolors=["tab:cyan","tab:olive","tab:green","tab:red","tab:gray","tab:pink","limegreen","tab:brown", "slateblue","gold"]
bencolors=["tab:cyan","tab:olive","tab:green","tab:red","tab:gray","tab:pink","limegreen","tab:brown", "gold","tab:purple"]

# -----select either original colors or ben's colors and order
modellist=modellist_ben
modelcolors=bencolors

latlist=['global','NHhighlat','NHmidlat','troplat','SHextratrop']

markerlist=['o','v','^','<','>','s','*','P','d','X']

linestyle_m = ['-','-.',':','--','-','-.','--',':','-','--']
mi=60

emissphase_namelist_plot=['Initial C \n [PgC]', 'Positive Emissions \n $\Delta$ C [PgC]','Net-zero Emissions \n$\Delta$ C [PgC]','Cumulative-zero Emissions\n $\Delta$ C [PgC]']
emissphase_namelist_table=['Initial C', 'Positive Emissions $\Delta$ C','Net-zero Emissions $\Delta$ C','Negative Emissions $\Delta$ C']


vegcolor=color=np.divide([176, 208, 65],255)
littercolor=np.divide([240, 155, 56],255)
soilcolor=np.divide([164, 119, 39],255)


# latcolors=['#94E86B','#6B74E8','#E8906B','red'] #
# latcolors=['#E8B46B','#6BE0E8','#6B91E8']
# latcolors=['#FE7180','#FFFA70','#70CFFF']
# latcolors=[vegcolor, littercolor,soilcolor]
 # latcolors=['#94E86B','#6B74E8','#E8906B']
# latcolors=['#E8B46B','#6BE0E8','#6B91E8']
# latcolors=['#FE7180','#FFFA70','#70CFFF']

pinkpurpleblue=['#EA7B7B','#7D589E','#C0D0DD'] #pink, purple, light blue
greenpurplebrown=['#B9CA60','#574F91','#D19563'] #green, purple, brown
vegsoil=[vegcolor, littercolor, soilcolor]

nutrientcolors=['#C0D0DD','#7D589E']  #lightblue #purple ] 
soilcolors    =['#574F91','#D19563'] #orange darkblue 
firecolors    =['#EA7B7B','#D3D3D3'] # grey red
dynvegcolors  =['#B9CA60','#574F91'] #green purple



latcolors=['#6B91E8', '#6BE0E8','#E8B46B']


subplotlabelnames=['a)','b)','c)','d)','e)','f)','g)','h)','i)','j)','k)','l)','m)','n)','o)','p)','q)','r)','s)','t)','u)','v)','w)','x)','y)','z)']

# choose one model to represent the 1 degree grid
model='GFDL-ESM4'
ds_all_grid=xr.open_dataset('processed_output/zonal_means_timepoints_' +model +'.nc')


# model structures
dynamicveg=[0, 0, 1, 0, 0, 1, 0, 1, 1, 0] 
soilpools=[0,1,1,0,1,0,0,0,0,0] #soilpools=[3,100,100,9,100,9,6,1,4,6]
nutrients=[1,1,0,0,1,1,0,0,1,1]
fire=[0,1,1,0,1,1,1,0,0,0]

In [56]:
#unitconversions
sperday=60*60*24
speryr=60*60*24*365

PgperKg = 1e-12
# to convert kg m-2 s-1 to kg/m2/yr multiply by speryr

## Load Data

In [57]:
# initialize a dictionary to hold grid data
data_dict={}
data_dict = load_grid(data_dict,modellist)

ACCESS-ESM1-5 getting grid info
CESM2 getting grid info
GFDL-ESM4 getting grid info
NASA-GISS-E2.1-G-CC2 getting grid info
NorESM2-LM getting grid info
MPI-ESM1-2-LR getting grid info
CNRM-ESM2-1 getting grid info
UKESM1.2 getting grid info
HadCM3LC-Bris getting grid info
MIROC-ES2L getting grid info


In [58]:
ds_C_global = xr.open_dataset("processed_output/C_metrics_matrix_fourboxSummer.nc")

In [59]:
ds_C_global

<xarray.Dataset> Size: 25MB
Dimensions:   (time: 4200, model: 10, run: 3, var: 5, latrange: 5)
Coordinates:
  * time      (time) int64 34kB 0 1 2 3 4 5 6 ... 4194 4195 4196 4197 4198 4199
  * model     (model) <U20 800B 'ACCESS-ESM1-5' 'CESM2' ... 'MIROC-ES2L'
  * run       (run) <U10 120B 'flat10' 'flat10_zec' 'flat10_cdr'
  * var       (var) <U7 140B 'cVeg' 'cSoil' 'cLitter' 'nbp' 'cTot'
  * latrange  (latrange) <U11 220B 'global' 'NHhighlat' ... 'SHextratrop'
Data variables:
    data      (time, model, run, var, latrange) float64 25MB ...

## Create tables of changes

### table by carbon pool

In [60]:
#-- table of changes in %


# rows are models
# columns are (%land, %litter, %soil, total [PgC])
# one set for each of (initial stocks, change during emissions, change after net zero, change after cumulative zero)



varlisthere=['cTot','cVeg','cLitter','cSoil']

# global only first
# l=0
for l in range(len(latlist)):
    initialC = np.empty([len(modellist),len(varlisthere)])
    emissC= np.empty([len(modellist),len(varlisthere)])
    AfterNZ= np.empty([len(modellist),len(varlisthere)])
    CumNZ= np.empty([len(modellist),len(varlisthere)])
    
    for m in range(len(modellist)):
        model=modellist[m]
        # mcolor=modelcolors[m]
        
        # for l in range(len(latlist)):
        
        for vi in range(len(varlisthere)):
            if vi==0:
                # total
                # vi=0
                e=0
                timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                           var=varlisthere[0], latrange=latlist[l])
                initC=np.mean(timeseries.data[0:9])
                nzC=np.mean(timeseries.data[90:109])
                e=1
                timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                           var=varlisthere[0], latrange=latlist[l])
                afternzC=np.mean(timeseries.data[190:199])
                e=2
                timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                           var=varlisthere[0], latrange=latlist[l])
                cumnz=np.mean(timeseries.data[190:199])
                
                initialC[m,vi]=initC
                emissC[m,vi]=nzC-initC
                AfterNZ[m,vi]=afternzC - nzC
                CumNZ[m,vi]=cumnz-initC

            else: 
                # (as % of total)
                e=0
                timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                           var=varlisthere[vi], latrange=latlist[l])
                initC=np.mean(timeseries.data[0:9])
                nzC=np.mean(timeseries.data[90:109])
                e=1
                timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                           var=varlisthere[vi], latrange=latlist[l])
                afternzC=np.mean(timeseries.data[190:199])
                e=2
                timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                           var=varlisthere[vi], latrange=latlist[l])
                cumnz=np.mean(timeseries.data[190:199])
                
                initialC[m,vi]=100*initC/initialC[m,0]
                emissC[m,vi]=100*(nzC-initC)/emissC[m,0]
                AfterNZ[m,vi]=100*(afternzC - nzC)/AfterNZ[m,0]
                CumNZ[m,vi]=100*(cumnz-initC)/CumNZ[m,0] 


    # add the multi model mean and standard deviation
    initialC=np.vstack((initialC,np.nanmean(initialC,0)))
    emissC=np.vstack((emissC,np.nanmean(emissC,0)))
    AfterNZ=np.vstack((AfterNZ,np.nanmean(AfterNZ,0)))
    CumNZ=np.vstack((CumNZ,np.nanmean(CumNZ,0)))
    
    initialC=np.vstack((initialC,np.nanstd(initialC,0)))
    emissC=np.vstack((emissC,np.nanstd(emissC,0)))
    AfterNZ=np.vstack((AfterNZ,np.nanstd(AfterNZ,0)))
    CumNZ=np.vstack((CumNZ,np.nanstd(CumNZ,0)))
    
    index_mdls= modellist.copy()
    index_mdls.append('Multi-model Mean')
    index_mdls.append('Multi-model Standard Dev.')
    
    
    
     # #-- add output to a pandas data frames
    column_names=["Total C [PgC]", "% Vegetation", "% Litter", "% Soil"]
    row_names = index_mdls
    data_arrays = [initialC]#, emissC, AfterNZ, CumNZ]
    group_names = ["Initial Carbon Content"]#,"Delta after Emissions Phase", "Following Net-Zero Emissions", "At Cumulative Zero Emissions"]
    # emissphase_namelist_table=['Initial C', 'Positive Emissions $\Delta$ C','Net-zero Emissions \Delta$ C','Negative Emissions $\Delta$ C']
    
    
    dfs = []
    
    for data, group in zip(data_arrays, group_names):
        df = pd.DataFrame(data, columns=column_names, index=row_names)
        df["Group"] = group
        dfs.append(df)
    
    # Combine into one DataFrame
    veg_stock_delta = pd.concat(dfs)

    veg_stock_delta.to_csv('tables/veg_stock_delta_' +latlist[l] +'_initialC.csv', index=False)

    #['global','NHhighlat','NHmidlat','troplat','SHextratrop']
    if l==0:
        veg_stock_delta_global=veg_stock_delta
    elif l==1:
        veg_stock_delta_NHhighlat=veg_stock_delta
    elif l==2:
        veg_stock_delta_NHmidlat=veg_stock_delta
    elif l==3:
        veg_stock_delta_troplat=veg_stock_delta
    else:
        veg_stock_delta_SHextratrop=veg_stock_delta

 

In [61]:
#-- table of changes - total Pg


# rows are models
# columns are (%land, %litter, %soil, total [PgC])
# one set for each of (initial stocks, change during emissions, change after net zero, change after cumulative zero)



varlisthere=['cTot','cVeg','cLitter','cSoil']

# global only first
# l=0
for l in range(len(latlist)):
    initialC = np.empty([len(modellist),len(varlisthere)])
    emissC= np.empty([len(modellist),len(varlisthere)])
    AfterNZ= np.empty([len(modellist),len(varlisthere)])
    CumNZ= np.empty([len(modellist),len(varlisthere)])
    
    for m in range(len(modellist)):
        model=modellist[m]
        # mcolor=modelcolors[m]
        
        # for l in range(len(latlist)):
        
        for vi in range(len(varlisthere)):
            if vi==0:
                # total
                # vi=0
                e=0
                timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                           var=varlisthere[0], latrange=latlist[l])
                initC=np.mean(timeseries.data[0:9])
                nzC=np.mean(timeseries.data[90:109])
                e=1
                timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                           var=varlisthere[0], latrange=latlist[l])
                afternzC=np.mean(timeseries.data[190:199])
                e=2
                timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                           var=varlisthere[0], latrange=latlist[l])
                cumnz=np.mean(timeseries.data[190:199])
                
                initialC[m,vi]=initC
                emissC[m,vi]=nzC-initC
                AfterNZ[m,vi]=afternzC - nzC
                CumNZ[m,vi]=cumnz-initC

            else: 
                # (as % of total)
                e=0
                timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                           var=varlisthere[vi], latrange=latlist[l])
                initC=np.mean(timeseries.data[0:9])
                nzC=np.mean(timeseries.data[90:109])
                e=1
                timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                           var=varlisthere[vi], latrange=latlist[l])
                afternzC=np.mean(timeseries.data[190:199])
                e=2
                timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                           var=varlisthere[vi], latrange=latlist[l])
                cumnz=np.mean(timeseries.data[190:199])
                
                initialC[m,vi]=initC
                emissC[m,vi]=(nzC-initC)
                AfterNZ[m,vi]=(afternzC - nzC)
                CumNZ[m,vi]=(cumnz-initC)


    # add the multi model mean and standard deviation
    initialC=np.vstack((initialC,np.nanmean(initialC,0)))
    emissC=np.vstack((emissC,np.nanmean(emissC,0)))
    AfterNZ=np.vstack((AfterNZ,np.nanmean(AfterNZ,0)))
    CumNZ=np.vstack((CumNZ,np.nanmean(CumNZ,0)))
    
    initialC=np.vstack((initialC,np.nanstd(initialC,0)))
    emissC=np.vstack((emissC,np.nanstd(emissC,0)))
    AfterNZ=np.vstack((AfterNZ,np.nanstd(AfterNZ,0)))
    CumNZ=np.vstack((CumNZ,np.nanstd(CumNZ,0)))
    
    index_mdls= modellist.copy()
    index_mdls.append('Multi-model Mean')
    index_mdls.append('Multi-model Standard Dev.')
    
    
    
     # #-- add output to a pandas data frames
    column_names=["Total C [PgC]", "Vegetation [PgC]", "Litter [PgC]", "Soil [PgC]"]
    row_names = index_mdls
    data_arrays = [initialC]#, emissC, AfterNZ, CumNZ]
    group_names = ["Initial Carbon Content"]#,"Delta after Emissions Phase", "Following Net-Zero Emissions", "At Cumulative Zero Emissions"]
    # emissphase_namelist_table=['Initial C', 'Positive Emissions $\Delta$ C','Net-zero Emissions \Delta$ C','Negative Emissions $\Delta$ C']
    
    
    dfs = []
    
    for data, group in zip(data_arrays, group_names):
        df = pd.DataFrame(data, columns=column_names, index=row_names)
        df["Group"] = group
        dfs.append(df)
    
    # Combine into one DataFrame
    veg_stock_delta = pd.concat(dfs)
    veg_stock_delta.to_csv('tables/veg_stock_delta_PgC_' +latlist[l] +'_initialC.csv', index=False)
    
    #['global','NHhighlat','NHmidlat','troplat','SHextratrop']
    if l==0:
        veg_stock_delta_PgC_global=veg_stock_delta
    elif l==1:
        veg_stock_delta_PgC_NHhighlat=veg_stock_delta
    elif l==2:
        veg_stock_delta_PgC_NHmidlat=veg_stock_delta
    elif l==3:
        veg_stock_delta_PgC_troplat=veg_stock_delta
    else:
        veg_stock_delta_PgC_SHextratrop=veg_stock_delta


    # if l==0:
    #     veg_stock_delta_PgC_global=veg_stock_delta
    # elif l==1:
    #     veg_stock_delta_PgC_highlat=veg_stock_delta
    # elif l==2:
    #     veg_stock_delta_PgC_midlat=veg_stock_delta
    # else:
    #     veg_stock_delta_PgC_troplat=veg_stock_delta


### Table by latitude

In [62]:
#-- table of changes for lat distribution


# rows are models
# columns are (global,highlat, midlat, lowlat)
# one set for each of (initial stocks, change during emissions, change after net zero, change after cumulative zero)



varlisthere=['cTot']#,'cVeg','cLitter','cSoil']

# global only first
# l=0
initialC = np.empty([len(modellist),len(latlist)])
emissC= np.empty([len(modellist),len(latlist)])
AfterNZ= np.empty([len(modellist),len(latlist)])
CumNZ= np.empty([len(modellist),len(latlist)])

for l in range(len(latlist)):
    for m in range(len(modellist)):
        model=modellist[m]
        # mcolor=modelcolors[m]
        
        # for l in range(len(latlist)):
        
        for vi in range(len(varlisthere)):

            e=0 # flat10
            timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                       var=varlisthere[0], latrange=latlist[l])
            initC=np.mean(timeseries.data[0:9])
            nzC=np.mean(timeseries.data[90:109])
            
            e=1 #flat10_zec
            timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                       var=varlisthere[0], latrange=latlist[l])
            # nzC=np.mean(timeseries.data[0:9])
            afternzC=np.mean(timeseries.data[190:199])
            
            e=2 #flat10_cdr
            timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                       var=varlisthere[0], latrange=latlist[l])
            cumnz=np.mean(timeseries.data[190:199])
            
            initialC[m,l]=initC
            emissC[m,l]=nzC-initC
            AfterNZ[m,l]=afternzC - nzC
            CumNZ[m,l]=cumnz-initC



# add the multi model mean and standard deviation
initialC=np.vstack((initialC,np.nanmean(initialC,0)))
emissC=np.vstack((emissC,np.nanmean(emissC,0)))
AfterNZ=np.vstack((AfterNZ,np.nanmean(AfterNZ,0)))
CumNZ=np.vstack((CumNZ,np.nanmean(CumNZ,0)))

initialC=np.vstack((initialC,np.nanstd(initialC,0)))
emissC=np.vstack((emissC,np.nanstd(emissC,0)))
AfterNZ=np.vstack((AfterNZ,np.nanstd(AfterNZ,0)))
CumNZ=np.vstack((CumNZ,np.nanstd(CumNZ,0)))

index_mdls= modellist.copy()
index_mdls.append('Multi-model Mean')
index_mdls.append('Multi-model Standard Dev.')



 # #-- add output to a pandas data frames
column_names=["Total C [PgC]", "NH Highlat C [PgC]", "NH Midlat C [PgC]", "Lowlat C [PgC]", "SH Extratropics [PgC]"] #['global','NHhighlat','NHmidlat','troplat','SHextratrop']
row_names = index_mdls
data_arrays = [initialC]#, emissC, AfterNZ, CumNZ]
group_names = ["Initial Carbon Content"]#,"Delta after Emissions Phase", "Following Net-Zero Emissions", "At Cumulative Zero Emissions"]

dfs = []

for data, group in zip(data_arrays, group_names):
    df = pd.DataFrame(data, columns=column_names, index=row_names)
    df["Group"] = group
    dfs.append(df)

# Combine into one DataFrame
veg_stock_delta = pd.concat(dfs)
veg_stock_delta.to_csv('tables/veg_stock_delta_' +varlisthere[0] +'_initialC.csv', index=False)

veg_stock_delta_cTot=veg_stock_delta



In [63]:
#-- table of changes for lat distribution


# rows are models
# columns are (global,highlat, midlat, lowlat)
# one set for each of (initial stocks, change during emissions, change after net zero, change after cumulative zero)



varlisthere=['cVeg']#,'cVeg','cLitter','cSoil']

# global only first
# l=0
initialC = np.empty([len(modellist),len(latlist)])
emissC= np.empty([len(modellist),len(latlist)])
AfterNZ= np.empty([len(modellist),len(latlist)])
CumNZ= np.empty([len(modellist),len(latlist)])

for l in range(len(latlist)):
    for m in range(len(modellist)):
        model=modellist[m]
        # mcolor=modelcolors[m]
        
        # for l in range(len(latlist)):
        
        for vi in range(len(varlisthere)):

            e=0 # flat10
            timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                       var=varlisthere[0], latrange=latlist[l])
            initC=np.mean(timeseries.data[0:9])
            nzC=np.mean(timeseries.data[90:109])
            
            e=1 #flat10_zec
            timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                       var=varlisthere[0], latrange=latlist[l])
            # nzC=np.mean(timeseries.data[0:9])
            afternzC=np.mean(timeseries.data[190:199])
            
            e=2 #flat10_cdr
            timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                       var=varlisthere[0], latrange=latlist[l])
            cumnz=np.mean(timeseries.data[190:199])
            
            initialC[m,l]=initC
            emissC[m,l]=nzC-initC
            AfterNZ[m,l]=afternzC - nzC
            CumNZ[m,l]=cumnz-initC



# add the multi model mean and standard deviation
initialC=np.vstack((initialC,np.nanmean(initialC,0)))
emissC=np.vstack((emissC,np.nanmean(emissC,0)))
AfterNZ=np.vstack((AfterNZ,np.nanmean(AfterNZ,0)))
CumNZ=np.vstack((CumNZ,np.nanmean(CumNZ,0)))

initialC=np.vstack((initialC,np.nanstd(initialC,0)))
emissC=np.vstack((emissC,np.nanstd(emissC,0)))
AfterNZ=np.vstack((AfterNZ,np.nanstd(AfterNZ,0)))
CumNZ=np.vstack((CumNZ,np.nanstd(CumNZ,0)))

index_mdls= modellist.copy()
index_mdls.append('Multi-model Mean')
index_mdls.append('Multi-model Standard Dev.')



 # #-- add output to a pandas data frames
column_names=["Total C [PgC]", "NH Highlat C [PgC]", "NH Midlat C [PgC]", "Lowlat C [PgC]", "SH Extratropics [PgC]"] #['global','NHhighlat','NHmidlat','troplat','SHextratrop']
row_names = index_mdls
data_arrays = [initialC]#, emissC, AfterNZ, CumNZ]
group_names = ["Initial Carbon Content"]#,"Delta after Emissions Phase", "Following Net-Zero Emissions", "At Cumulative Zero Emissions"]

dfs = []

for data, group in zip(data_arrays, group_names):
    df = pd.DataFrame(data, columns=column_names, index=row_names)
    df["Group"] = group
    dfs.append(df)

# Combine into one DataFrame
veg_stock_delta = pd.concat(dfs)
veg_stock_delta.to_csv('tables/veg_stock_delta_' +varlisthere[0] +'_initialC.csv', index=False)

veg_stock_delta_cVeg = veg_stock_delta

In [64]:
#-- table of changes for lat distribution


# rows are models
# columns are (global,highlat, midlat, lowlat)
# one set for each of (initial stocks, change during emissions, change after net zero, change after cumulative zero)



varlisthere=['cSoil']#,'cVeg','cLitter','cSoil']

# global only first
# l=0
initialC = np.empty([len(modellist),len(latlist)])
emissC= np.empty([len(modellist),len(latlist)])
AfterNZ= np.empty([len(modellist),len(latlist)])
CumNZ= np.empty([len(modellist),len(latlist)])

for l in range(len(latlist)):
    for m in range(len(modellist)):
        model=modellist[m]
        # mcolor=modelcolors[m]
        
        # for l in range(len(latlist)):
        
        for vi in range(len(varlisthere)):

            e=0 # flat10
            timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                       var=varlisthere[0], latrange=latlist[l])
            initC=np.mean(timeseries.data[0:9])
            nzC=np.mean(timeseries.data[90:109])
            
            e=1 #flat10_zec
            timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                       var=varlisthere[0], latrange=latlist[l])
            # nzC=np.mean(timeseries.data[0:9])
            afternzC=np.mean(timeseries.data[190:199])
            
            e=2 #flat10_cdr
            timeseries=ds_C_global.sel(model=modellist[m], run=runlist[e], 
                                       var=varlisthere[0], latrange=latlist[l])
            cumnz=np.mean(timeseries.data[190:199])
            
            initialC[m,l]=initC
            emissC[m,l]=nzC-initC
            AfterNZ[m,l]=afternzC - nzC
            CumNZ[m,l]=cumnz-initC



# add the multi model mean and standard deviation
initialC=np.vstack((initialC,np.nanmean(initialC,0)))
emissC=np.vstack((emissC,np.nanmean(emissC,0)))
AfterNZ=np.vstack((AfterNZ,np.nanmean(AfterNZ,0)))
CumNZ=np.vstack((CumNZ,np.nanmean(CumNZ,0)))

initialC=np.vstack((initialC,np.nanstd(initialC,0)))
emissC=np.vstack((emissC,np.nanstd(emissC,0)))
AfterNZ=np.vstack((AfterNZ,np.nanstd(AfterNZ,0)))
CumNZ=np.vstack((CumNZ,np.nanstd(CumNZ,0)))

index_mdls= modellist.copy()
index_mdls.append('Multi-model Mean')
index_mdls.append('Multi-model Standard Dev.')



 # #-- add output to a pandas data frames
column_names=["Total C [PgC]", "NH Highlat C [PgC]", "NH Midlat C [PgC]", "Lowlat C [PgC]", "SH Extratropics [PgC]"] #['global','NHhighlat','NHmidlat','troplat','SHextratrop']
row_names = index_mdls
data_arrays = [initialC]#, emissC, AfterNZ, CumNZ]
group_names = ["Initial Carbon Content"]#,"Delta after Emissions Phase", "Following Net-Zero Emissions", "At Cumulative Zero Emissions"]

dfs = []

for data, group in zip(data_arrays, group_names):
    df = pd.DataFrame(data, columns=column_names, index=row_names)
    df["Group"] = group
    dfs.append(df)

# Combine into one DataFrame
veg_stock_delta = pd.concat(dfs)
veg_stock_delta.to_csv('tables/veg_stock_delta_' +varlisthere[0] +'_initialC.csv', index=False)

veg_stock_delta_cSoil = veg_stock_delta


# Print tables to screen

In [65]:
veg_stock_delta_cTot

,Total C [PgC],NH Highlat C [PgC],NH Midlat C [PgC],Lowlat C [PgC],SH Extratropics [PgC],Group
ACCESS-ESM1-5,1512.212970,465.857763,493.864199,399.756978,646.598207,Initial Carbon Content
CESM2,1983.182821,974.896250,576.294945,292.429513,715.857057,Initial Carbon Content
GFDL-ESM4,972.546212,416.849486,300.254812,203.644211,352.052525,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,2265.570435,1012.706972,519.939999,590.007774,662.855709,Initial Carbon Content
NorESM2-LM,3116.579871,1986.059429,577.068558,282.602500,847.917942,Initial Carbon Content
MPI-ESM1-2-LR,1382.143918,378.561037,544.549167,338.846121,664.736771,Initial Carbon Content
CNRM-ESM2-1,2165.395540,788.165783,555.311234,626.414441,750.815269,Initial Carbon Content
UKESM1.2,2333.319736,550.131156,830.240275,710.069362,1073.119232,Initial Carbon Content
HadCM3LC-Bris,1991.894124,629.215442,726.492747,442.862860,919.815826,Initial Carbon Content
MIROC-ES2L,1940.083133,706.109923,587.806174,492.796864,741.176350,Initial Carbon Content


In [66]:
veg_stock_delta_cSoil

,Total C [PgC],NH Highlat C [PgC],NH Midlat C [PgC],Lowlat C [PgC],SH Extratropics [PgC],Group
ACCESS-ESM1-5,824.180881,318.316349,170.718886,248.226730,257.637790,Initial Carbon Content
CESM2,1400.462104,847.416361,257.395260,201.690070,351.355676,Initial Carbon Content
GFDL-ESM4,331.870558,172.601115,63.816651,78.429913,80.839534,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,1553.356066,864.441277,235.665059,383.647118,305.267687,Initial Carbon Content
NorESM2-LM,2549.201493,1866.110534,253.442996,197.142380,485.948579,Initial Carbon Content
MPI-ESM1-2-LR,683.098680,237.751136,196.642487,195.202482,250.145070,Initial Carbon Content
CNRM-ESM2-1,1442.745605,615.964905,290.497765,406.934072,419.846578,Initial Carbon Content
UKESM1.2,1741.026706,464.161455,498.148051,594.946513,681.918735,Initial Carbon Content
HadCM3LC-Bris,1345.322456,541.036662,314.589257,371.938334,432.347460,Initial Carbon Content
MIROC-ES2L,1133.977824,527.178596,240.329263,294.041955,312.757273,Initial Carbon Content


In [67]:
veg_stock_delta_cVeg

,Total C [PgC],NH Highlat C [PgC],NH Midlat C [PgC],Lowlat C [PgC],SH Extratropics [PgC],Group
ACCESS-ESM1-5,644.864095,132.456194,312.043209,138.743049,373.664842,Initial Carbon Content
CESM2,550.242276,113.423530,310.402435,83.510386,353.308356,Initial Carbon Content
GFDL-ESM4,434.509810,137.158756,183.997375,94.338675,203.012383,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,712.214369,148.265695,284.274940,206.360655,357.588023,Initial Carbon Content
NorESM2-LM,537.895141,107.889406,314.789156,79.211705,350.794030,Initial Carbon Content
MPI-ESM1-2-LR,426.735297,49.381598,255.482171,77.187358,300.166348,Initial Carbon Content
CNRM-ESM2-1,508.918837,86.902841,216.521074,160.523266,261.492727,Initial Carbon Content
UKESM1.2,592.293030,85.969701,332.092224,115.122849,391.200497,Initial Carbon Content
HadCM3LC-Bris,646.571669,88.178780,411.903490,70.924526,487.468367,Initial Carbon Content
MIROC-ES2L,522.731669,78.735013,270.402317,121.655109,322.341546,Initial Carbon Content


In [68]:
veg_stock_delta_global

,Total C [PgC],% Vegetation,% Litter,% Soil,Group
ACCESS-ESM1-5,1512.212970,42.643735,2.854624,54.501641,Initial Carbon Content
CESM2,1983.182821,27.745414,1.637693,70.616894,Initial Carbon Content
GFDL-ESM4,972.546212,44.677549,21.198565,34.123886,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,2265.570435,31.436426,NaN,68.563574,Initial Carbon Content
NorESM2-LM,3116.579871,17.259148,0.946013,81.794839,Initial Carbon Content
MPI-ESM1-2-LR,1382.143918,30.874882,19.701996,49.423122,Initial Carbon Content
CNRM-ESM2-1,2165.395540,23.502350,9.870303,66.627347,Initial Carbon Content
UKESM1.2,2333.319736,25.384135,NaN,74.615865,Initial Carbon Content
HadCM3LC-Bris,1991.894124,32.460142,NaN,67.539858,Initial Carbon Content
MIROC-ES2L,1940.083133,26.943777,14.606263,58.449960,Initial Carbon Content


In [69]:
veg_stock_delta_NHhighlat

,Total C [PgC],% Vegetation,% Litter,% Soil,Group
ACCESS-ESM1-5,465.857763,28.432754,3.238160,68.329085,Initial Carbon Content
CESM2,974.896250,11.634421,1.441831,86.923748,Initial Carbon Content
GFDL-ESM4,416.849486,32.903664,25.690236,41.406100,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,1012.706972,14.640533,NaN,85.359467,Initial Carbon Content
NorESM2-LM,1986.059429,5.432335,0.607207,93.960458,Initial Carbon Content
MPI-ESM1-2-LR,378.561037,13.044554,24.151535,62.803911,Initial Carbon Content
CNRM-ESM2-1,788.165783,11.025960,10.822347,78.151693,Initial Carbon Content
UKESM1.2,550.131156,15.627128,NaN,84.372872,Initial Carbon Content
HadCM3LC-Bris,629.215442,14.014084,NaN,85.985916,Initial Carbon Content
MIROC-ES2L,706.109923,11.150532,14.189903,74.659565,Initial Carbon Content


In [70]:
veg_stock_delta_NHmidlat

,Total C [PgC],% Vegetation,% Litter,% Soil,Group
ACCESS-ESM1-5,493.864199,63.184011,2.248007,34.567982,Initial Carbon Content
CESM2,576.294945,53.861731,1.474462,44.663807,Initial Carbon Content
GFDL-ESM4,300.254812,61.280409,17.465427,21.254164,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,519.939999,54.674566,NaN,45.325434,Initial Carbon Content
NorESM2-LM,577.068558,54.549698,1.531258,43.919044,Initial Carbon Content
MPI-ESM1-2-LR,544.549167,46.916272,16.972666,36.111062,Initial Carbon Content
CNRM-ESM2-1,555.311234,38.990941,8.696455,52.312604,Initial Carbon Content
UKESM1.2,830.240275,39.999532,NaN,60.000468,Initial Carbon Content
HadCM3LC-Bris,726.492747,56.697536,NaN,43.302464,Initial Carbon Content
MIROC-ES2L,587.806174,46.001952,13.112247,40.885801,Initial Carbon Content


In [71]:
veg_stock_delta_troplat

,Total C [PgC],% Vegetation,% Litter,% Soil,Group
ACCESS-ESM1-5,399.756978,34.706849,3.198743,62.094408,Initial Carbon Content
CESM2,292.429513,28.557441,2.472068,68.970491,Initial Carbon Content
GFDL-ESM4,203.644211,46.325242,15.161552,38.513205,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,590.007774,34.975921,NaN,65.024079,Initial Carbon Content
NorESM2-LM,282.602500,28.029372,2.211026,69.759602,Initial Carbon Content
MPI-ESM1-2-LR,338.846121,22.779472,19.612525,57.608003,Initial Carbon Content
CNRM-ESM2-1,626.414441,25.625729,9.411836,64.962435,Initial Carbon Content
UKESM1.2,710.069362,16.212902,NaN,83.787098,Initial Carbon Content
HadCM3LC-Bris,442.862860,16.015009,NaN,83.984991,Initial Carbon Content
MIROC-ES2L,492.796864,24.686665,15.645351,59.667984,Initial Carbon Content


In [72]:
veg_stock_delta_SHextratrop

,Total C [PgC],% Vegetation,% Litter,% Soil,Group
ACCESS-ESM1-5,646.598207,57.789341,2.365545,39.845114,Initial Carbon Content
CESM2,715.857057,49.354596,1.563584,49.081820,Initial Carbon Content
GFDL-ESM4,352.052525,57.665368,19.372282,22.962350,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,662.855709,53.946586,NaN,46.053414,Initial Carbon Content
NorESM2-LM,847.917942,41.371224,1.317973,57.310803,Initial Carbon Content
MPI-ESM1-2-LR,664.736771,45.155671,17.213634,37.630696,Initial Carbon Content
CNRM-ESM2-1,750.815269,34.827838,9.253403,55.918759,Initial Carbon Content
UKESM1.2,1073.119232,36.454523,NaN,63.545477,Initial Carbon Content
HadCM3LC-Bris,919.815826,52.996301,NaN,47.003699,Initial Carbon Content
MIROC-ES2L,741.176350,43.490533,14.312050,42.197417,Initial Carbon Content


In [73]:
veg_stock_delta_PgC_global

,Total C [PgC],Vegetation [PgC],Litter [PgC],Soil [PgC],Group
ACCESS-ESM1-5,1512.212970,644.864095,43.167994,824.180881,Initial Carbon Content
CESM2,1983.182821,550.242276,32.478441,1400.462104,Initial Carbon Content
GFDL-ESM4,972.546212,434.509810,206.165844,331.870558,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,2265.570435,712.214369,NaN,1553.356066,Initial Carbon Content
NorESM2-LM,3116.579871,537.895141,29.483236,2549.201493,Initial Carbon Content
MPI-ESM1-2-LR,1382.143918,426.735297,272.309940,683.098680,Initial Carbon Content
CNRM-ESM2-1,2165.395540,508.918837,213.731098,1442.745605,Initial Carbon Content
UKESM1.2,2333.319736,592.293030,NaN,1741.026706,Initial Carbon Content
HadCM3LC-Bris,1991.894124,646.571669,NaN,1345.322456,Initial Carbon Content
MIROC-ES2L,1940.083133,522.731669,283.373640,1133.977824,Initial Carbon Content


In [74]:
veg_stock_delta_PgC_NHhighlat

,Total C [PgC],Vegetation [PgC],Litter [PgC],Soil [PgC],Group
ACCESS-ESM1-5,465.857763,132.456194,15.085221,318.316349,Initial Carbon Content
CESM2,974.896250,113.423530,14.056359,847.416361,Initial Carbon Content
GFDL-ESM4,416.849486,137.158756,107.089615,172.601115,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,1012.706972,148.265695,NaN,864.441277,Initial Carbon Content
NorESM2-LM,1986.059429,107.889406,12.059489,1866.110534,Initial Carbon Content
MPI-ESM1-2-LR,378.561037,49.381598,91.428303,237.751136,Initial Carbon Content
CNRM-ESM2-1,788.165783,86.902841,85.298038,615.964905,Initial Carbon Content
UKESM1.2,550.131156,85.969701,NaN,464.161455,Initial Carbon Content
HadCM3LC-Bris,629.215442,88.178780,NaN,541.036662,Initial Carbon Content
MIROC-ES2L,706.109923,78.735013,100.196314,527.178596,Initial Carbon Content


In [75]:
veg_stock_delta_PgC_NHmidlat

,Total C [PgC],Vegetation [PgC],Litter [PgC],Soil [PgC],Group
ACCESS-ESM1-5,493.864199,312.043209,11.102104,170.718886,Initial Carbon Content
CESM2,576.294945,310.402435,8.497250,257.395260,Initial Carbon Content
GFDL-ESM4,300.254812,183.997375,52.440786,63.816651,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,519.939999,284.274940,NaN,235.665059,Initial Carbon Content
NorESM2-LM,577.068558,314.789156,8.836406,253.442996,Initial Carbon Content
MPI-ESM1-2-LR,544.549167,255.482171,92.424510,196.642487,Initial Carbon Content
CNRM-ESM2-1,555.311234,216.521074,48.292394,290.497765,Initial Carbon Content
UKESM1.2,830.240275,332.092224,NaN,498.148051,Initial Carbon Content
HadCM3LC-Bris,726.492747,411.903490,NaN,314.589257,Initial Carbon Content
MIROC-ES2L,587.806174,270.402317,77.074595,240.329263,Initial Carbon Content


In [76]:
veg_stock_delta_PgC_troplat

,Total C [PgC],Vegetation [PgC],Litter [PgC],Soil [PgC],Group
ACCESS-ESM1-5,399.756978,138.743049,12.787199,248.226730,Initial Carbon Content
CESM2,292.429513,83.510386,7.229057,201.690070,Initial Carbon Content
GFDL-ESM4,203.644211,94.338675,30.875624,78.429913,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,590.007774,206.360655,NaN,383.647118,Initial Carbon Content
NorESM2-LM,282.602500,79.211705,6.248415,197.142380,Initial Carbon Content
MPI-ESM1-2-LR,338.846121,77.187358,66.456280,195.202482,Initial Carbon Content
CNRM-ESM2-1,626.414441,160.523266,58.957103,406.934072,Initial Carbon Content
UKESM1.2,710.069362,115.122849,NaN,594.946513,Initial Carbon Content
HadCM3LC-Bris,442.862860,70.924526,NaN,371.938334,Initial Carbon Content
MIROC-ES2L,492.796864,121.655109,77.099800,294.041955,Initial Carbon Content


In [77]:
veg_stock_delta_PgC_SHextratrop

,Total C [PgC],Vegetation [PgC],Litter [PgC],Soil [PgC],Group
ACCESS-ESM1-5,646.598207,373.664842,15.295574,257.637790,Initial Carbon Content
CESM2,715.857057,353.308356,11.193025,351.355676,Initial Carbon Content
GFDL-ESM4,352.052525,203.012383,68.200607,80.839534,Initial Carbon Content
NASA-GISS-E2.1-G-CC2,662.855709,357.588023,NaN,305.267687,Initial Carbon Content
NorESM2-LM,847.917942,350.794030,11.175332,485.948579,Initial Carbon Content
MPI-ESM1-2-LR,664.736771,300.166348,114.425353,250.145070,Initial Carbon Content
CNRM-ESM2-1,750.815269,261.492727,69.475964,419.846578,Initial Carbon Content
UKESM1.2,1073.119232,391.200497,NaN,681.918735,Initial Carbon Content
HadCM3LC-Bris,919.815826,487.468367,NaN,432.347460,Initial Carbon Content
MIROC-ES2L,741.176350,322.341546,106.077531,312.757273,Initial Carbon Content
